In [ ]:
import os, shutil, subprocess, sys

# ── Paths ──
CODE_BASE    = "/kaggle/input/datasets/neymaschine/codeee"
CHEXPERT_BASE = "/kaggle/input/datasets/ashery/chexpert"
WORK         = "/kaggle/working"

# ── Copy code to working dir ──
for item in ("src", "configs"):
    dst = os.path.join(WORK, item)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(os.path.join(CODE_BASE, item), dst)

# ── Install deps ──
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "pyyaml", "tensorboard"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchxrayvision"])

os.chdir(WORK)
print("Setup done. CWD:", os.getcwd())
print("Code files:", os.listdir(os.path.join(WORK, "src")))
print("Configs:", os.listdir(os.path.join(WORK, "configs")))

In [ ]:
import os, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

WORK = "/kaggle/working"

# ── Find CheXpert train.csv ──
CANDIDATES = [
    "/kaggle/input/datasets/ashery/chexpert/train.csv",
    "/kaggle/input/datasets/ashery/chexpert/CheXpert-v1.0-small/train.csv",
    "/kaggle/input/chexpert/CheXpert-v1.0-small/train.csv",
]

orig_train = None
for c in CANDIDATES:
    if os.path.exists(c):
        orig_train = c
        break

if orig_train is None:
    # Auto-discover
    import subprocess
    result = subprocess.run(
        ["find", "/kaggle/input", "-name", "train.csv"],
        capture_output=True, text=True
    )
    print("Found train.csv files:")
    print(result.stdout)
    raise FileNotFoundError("Cannot find train.csv — check the path above and update CANDIDATES")

print(f"Using: {orig_train}")
df = pd.read_csv(orig_train)
print(f"Loaded {len(df)} rows")

# ── Split train → train_split (85%) + test (15%) ──
df["patient_id"] = df["Path"].str.extract(r"(patient\d+)")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["patient_id"]))
df_train = df.iloc[train_idx].drop(columns=["patient_id"])
df_test  = df.iloc[test_idx].drop(columns=["patient_id"])
df_train.to_csv(os.path.join(WORK, "train_split.csv"), index=False)
df_test.to_csv(os.path.join(WORK, "test.csv"), index=False)
print(f"train_split.csv: {len(df_train)} rows, test.csv: {len(df_test)} rows")

# ── Split train_split → train_proper (95%) + val_proper (5%) ──
df_train["patient_id"] = df_train["Path"].str.extract(r"(patient\d+)")
splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=42)
train_idx2, val_idx2 = next(splitter2.split(df_train, groups=df_train["patient_id"]))
df_train_proper = df_train.iloc[train_idx2].drop(columns=["patient_id"])
df_val_proper   = df_train.iloc[val_idx2].drop(columns=["patient_id"])
df_train_proper.to_csv(os.path.join(WORK, "train_proper.csv"), index=False)
df_val_proper.to_csv(os.path.join(WORK, "val_proper.csv"), index=False)
print(f"train_proper.csv: {len(df_train_proper)} rows, val_proper.csv: {len(df_val_proper)} rows")

# ── Verify ──
for f in ["train_proper.csv", "val_proper.csv", "test.csv"]:
    full = os.path.join(WORK, f)
    print(f"{'✓' if os.path.exists(full) else '✗'} {full}")

In [ ]:
import os, yaml

CHEXPERT_BASE = "/kaggle/input/datasets/ashery/chexpert"
WORK = "/kaggle/working"

swin_config = f"""experiment_name: swin_tiny

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 384
  frontal_only: true
  uncertainty_strategy: u-mask

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: swin_tiny
  pretrained: true
  num_classes: 5

training:
  batch_size: 32
  num_workers: 2
  epochs: 30
  learning_rate: 5.0e-05
  weight_decay: 0.05
  warmup_epochs: 2
  scheduler: cosine
  early_stopping_patience: 7

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

vit_config = f"""experiment_name: vit_base

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mask

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: vit_base
  pretrained: true
  num_classes: 5

training:
  batch_size: 64
  num_workers: 2
  epochs: 30
  learning_rate: 3.0e-05
  weight_decay: 0.1
  warmup_epochs: 2
  scheduler: cosine
  early_stopping_patience: 7

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

densenet_cxr_config = f"""experiment_name: densenet121_cxr

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mask
  cxr_pretrained: true

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: densenet121_cxr
  pretrained: true
  num_classes: 5

training:
  batch_size: 32
  num_workers: 2
  epochs: 15
  learning_rate: 1.0e-04
  weight_decay: 1.0e-05
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 5

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

consolidation_config = f"""experiment_name: swin_tiny_consolidation

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mask

labels:
  target_labels:
  - Consolidation

model:
  architecture: swin_tiny
  pretrained: true
  num_classes: 1

training:
  batch_size: 128
  num_workers: 2
  epochs: 30
  learning_rate: 5.0e-05
  weight_decay: 0.05
  warmup_epochs: 5
  scheduler: cosine
  early_stopping_patience: 7

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 008 (DONE — test AUROC 0.8129)
swin_perclass_config = f"""experiment_name: swin_tiny_perclass

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: swin_tiny
  pretrained: true
  num_classes: 5

training:
  batch_size: 64
  num_workers: 2
  epochs: 30
  learning_rate: 5.0e-05
  weight_decay: 0.05
  warmup_epochs: 2
  scheduler: cosine
  early_stopping_patience: 7
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 009 (DONE — test AUROC 0.8118)
vit_perclass_config = f"""experiment_name: vit_base_perclass

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: vit_base
  pretrained: true
  num_classes: 5

training:
  batch_size: 64
  num_workers: 2
  epochs: 30
  learning_rate: 3.0e-05
  weight_decay: 0.1
  warmup_epochs: 2
  scheduler: cosine
  early_stopping_patience: 7
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 010 (DONE — test AUROC 0.8298)
densenet_perclass_config = f"""experiment_name: densenet121_perclass

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: densenet121
  pretrained: true
  num_classes: 5

training:
  batch_size: 64
  num_workers: 2
  epochs: 20
  learning_rate: 1.0e-04
  weight_decay: 0.01
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 5
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 011: same as Run 010 but patience=7 (ablation: was early stopping too aggressive?)
densenet_perclass_longer_config = f"""experiment_name: densenet121_perclass_longer

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion

model:
  architecture: densenet121
  pretrained: true
  num_classes: 5

training:
  batch_size: 64
  num_workers: 2
  epochs: 20
  learning_rate: 1.0e-04
  weight_decay: 0.01
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 7
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 012: DenseNet121 with 7 labels — DONE (test AUROC 0.8269 / 7-label mean)
densenet_7labels_config = f"""experiment_name: densenet121_7labels

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros
    Fracture: u-ones
    Pneumothorax: u-ones

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion
  - Fracture
  - Pneumothorax

model:
  architecture: densenet121
  pretrained: true
  num_classes: 7

training:
  batch_size: 64
  num_workers: 2
  epochs: 20
  learning_rate: 1.0e-04
  weight_decay: 0.01
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 5
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 013: DenseNet121 7-labels v2 — nan-mask for Fracture & Pneumothorax
# Root cause of Fracture F1=0.22: 94.5% NaN treated as implicit-negative flooded
# training with 170,405 false negatives (22:1 imbalance). nan-mask excludes all
# unlabelled rows from the Fracture/Pneumothorax loss so only the 9,836 / 63,716
# explicitly labelled rows contribute. Expected: Fracture F1 >> 0.50.
densenet_7labels_v2_config = f"""experiment_name: densenet121_7labels_v2

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros
    Fracture: nan-mask
    Pneumothorax: nan-mask

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion
  - Fracture
  - Pneumothorax

model:
  architecture: densenet121
  pretrained: true
  num_classes: 7

training:
  batch_size: 64
  num_workers: 2
  epochs: 20
  learning_rate: 1.0e-04
  weight_decay: 0.01
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 5
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

# Run 014: DenseNet121 6-labels — drop Fracture (unusable F1), keep Pneumothorax
# Rationale: Fracture AUROC 0.3988 in Run 013 / F1-opt 0.2150 in Run 012 means
# it's unsuitable for user study. Dropping it should recover 5-label performance
# toward Run 010 levels (~0.83) while retaining Pneumothorax (~0.88) for
# incidental finding detection.
densenet_6labels_config = f"""experiment_name: densenet121_6labels

data:
  data_dir: {CHEXPERT_BASE}
  train_csv: {WORK}/train_proper.csv
  valid_csv: {WORK}/val_proper.csv
  test_csv: {WORK}/test.csv
  image_size: 224
  frontal_only: true
  uncertainty_strategy: u-mixed
  per_label_strategies:
    Cardiomegaly: u-zeros
    Edema: u-ones
    Consolidation: u-ones
    Atelectasis: u-ones
    Pleural Effusion: u-zeros
    Pneumothorax: u-ones

labels:
  target_labels:
  - Cardiomegaly
  - Edema
  - Consolidation
  - Atelectasis
  - Pleural Effusion
  - Pneumothorax

model:
  architecture: densenet121
  pretrained: true
  num_classes: 6

training:
  batch_size: 64
  num_workers: 2
  epochs: 20
  learning_rate: 1.0e-04
  weight_decay: 0.01
  warmup_epochs: 1
  scheduler: cosine
  early_stopping_patience: 5
  pos_weight: true

device: auto

logging:
  checkpoint_dir: {WORK}/checkpoints
  log_dir: {WORK}/runs
  save_best_only: true
"""

configs = {
    "swin_tiny.yaml": swin_config,
    "vit_base.yaml": vit_config,
    "densenet121_cxr.yaml": densenet_cxr_config,
    "swin_tiny_consolidation.yaml": consolidation_config,
    "swin_tiny_perclass.yaml": swin_perclass_config,
    "vit_base_perclass.yaml": vit_perclass_config,
    "densenet121_perclass.yaml": densenet_perclass_config,
    "densenet121_perclass_longer.yaml": densenet_perclass_longer_config,
    "densenet121_7labels.yaml": densenet_7labels_config,
    "densenet121_7labels_v2.yaml": densenet_7labels_v2_config,
    "densenet121_6labels.yaml": densenet_6labels_config,
}

for fname, content in configs.items():
    path = os.path.join(WORK, "configs", fname)
    with open(path, "w") as f:
        f.write(content)
    with open(path) as f:
        check = yaml.safe_load(f)
    print(f"✓ {fname} → image_size: {check['data']['image_size']}, arch: {check['model']['architecture']}, classes: {check['model']['num_classes']}")


In [ ]:
import timm, os

print("Pre-downloading pretrained weights...")

# Download Swin-Tiny weights
m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=5)
del m
print("✓ swin_tiny downloaded")

# Download ViT-Base weights
m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=5)
del m
print("✓ vit_base downloaded")

import torchxrayvision as xrv
m = xrv.models.DenseNet(weights="densenet121-res224-all")
del m
print("✓ densenet121_cxr (torchxrayvision) downloaded")

# Check cache location
cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(cache):
    for f in os.listdir(cache):
        print(f"  cached: {f}")
print("All weights ready.")

In [ ]:
skip =True

if not skip: 
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)
    
    print("=" * 60)
    print("  TRAINING: Swin-Tiny")
    print("=" * 60)
    
    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/swin_tiny.yaml"
    ], check=True)
    
    # Immediate backup
    src_ckpt = os.path.join(WORK, "checkpoints", "swin_tiny", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "swin_tiny_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")
    
    print("\n" + "=" * 60)
    print("  EVALUATING: Swin-Tiny on TEST set")
    print("=" * 60)
    
    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/swin_tiny.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)

In [ ]:
skip = True

if not skip: 
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)
    
    print("=" * 60)
    print("  TRAINING: ViT-Base")
    print("=" * 60)
    
    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/vit_base.yaml"
    ], check=True)
    
    src_ckpt = os.path.join(WORK, "checkpoints", "vit_base", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "vit_base_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")
    
    print("\n" + "=" * 60)
    print("  EVALUATING: ViT-Base on TEST set")
    print("=" * 60)
    
    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/vit_base.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)

In [ ]:
skip = True

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121-CXR")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_cxr.yaml"
    ], check=True)

    # Immediate backup
    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_cxr", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_cxr_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121-CXR on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_cxr.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test",
        "--tta"
    ], check=True)

In [ ]:
skip = True

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: Swin-Tiny (Consolidation only)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/swin_tiny_consolidation.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "swin_tiny_consolidation", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "swin_tiny_consolidation_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: Swin-Tiny (Consolidation) on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/swin_tiny_consolidation.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)

In [ ]:
skip = True  # Run 008: DONE — test AUROC 0.8129

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: Swin-Tiny (per-label U-strategy + pos_weight)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/swin_tiny_perclass.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "swin_tiny_perclass", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "swin_tiny_perclass_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: Swin-Tiny perclass on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/swin_tiny_perclass.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)


In [ ]:
skip = True  # Run 009: DONE — test AUROC 0.8118

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: ViT-Base (per-label U-strategy + pos_weight)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/vit_base_perclass.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "vit_base_perclass", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "vit_base_perclass_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: ViT-Base perclass on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/vit_base_perclass.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)


In [ ]:
skip = True  # Run 010: DONE — test AUROC 0.8298

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121 (per-label U-strategy + pos_weight)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_perclass.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_perclass", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_perclass_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121 perclass on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_perclass.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)


In [ ]:
skip = True  # Run 011: DenseNet121 perclass — patience=7 ablation (vs Run 010 patience=5)

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121 perclass longer (patience=7)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_perclass_longer.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_perclass_longer", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_perclass_longer_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121 perclass longer on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_perclass_longer.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)


In [ ]:
skip = True  # Run 012: DONE — test AUROC 0.8269 (7-label mean) / 0.8073 (5-label mean)

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121 7-labels (Primary/Incidental)")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_7labels.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_7labels", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_7labels_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121 7-labels on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_7labels.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)


In [ ]:
skip = True  # Run 013: DONE — test AUROC 0.7623 (7-label) / 0.8180 (5-label). Fracture collapsed to 0.3988.

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121 7-labels v2 (nan-mask fix)")
    print("  Fracture & Pneumothorax: NaN → masked (not implicit-negative)")
    print("  Fracture effective imbalance: 22:1 → 0.28:1")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_7labels_v2.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_7labels_v2", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_7labels_v2_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")
    else:
        print(f"WARNING: checkpoint not found at {src_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121 7-labels v2 on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_7labels_v2.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)

In [ ]:
skip = True  # Run 014: DONE — test AUROC 0.8361 (6-label mean) / 0.8211 (5-label mean)

if not skip:
    import subprocess, sys, shutil, os
    WORK = "/kaggle/working"
    os.chdir(WORK)

    print("=" * 60)
    print("  TRAINING: DenseNet121 6-labels (5 core + Pneumothorax)")
    print("  Fracture dropped — unusable F1 in Runs 012/013")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.train",
        "--config", "configs/densenet121_6labels.yaml"
    ], check=True)

    src_ckpt = os.path.join(WORK, "checkpoints", "densenet121_6labels", "best_model.pt")
    dst_ckpt = os.path.join(WORK, "densenet121_6labels_best_model.pt")
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f"\n>>> Saved: {dst_ckpt}")
    else:
        print(f"WARNING: checkpoint not found at {src_ckpt}")

    print("\n" + "=" * 60)
    print("  EVALUATING: DenseNet121 6-labels on TEST set")
    print("=" * 60)

    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--config", "configs/densenet121_6labels.yaml",
        "--checkpoint", dst_ckpt,
        "--split", "test"
    ], check=True)